# ML Zoomcamp 2026 - Module 3 Homework

In [17]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score
from sklearn.model_selection import train_test_split

In [18]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation

Fill missing categorical values with `NA` and numerical values with `0.0`.

In [19]:
categorical = [
    'lead_source',
    'industry',
    'employment_status',
    'location',
]

numerical = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score',
]

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

## Question 1

What is the most frequent observation for `industry`?

- `NA`
- `technology`
- `healthcare`
- `retail`

In [20]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

## Question 2

Which pair has the biggest correlation?

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

In [21]:
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


## Split the data

Use the exact split from the homework and remove `converted` from the feature dataframes.

In [22]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

## Question 3

Calculate mutual information between `converted` and each categorical variable using the training set. Round each score to 2 decimals.

Which variable has the biggest score?

- `industry`
- `location`
- `lead_source`
- `employment_status`

In [23]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical].apply(mutual_info_converted_score)
mi.round(2).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

## Question 4

One-hot encode the features, train the specified logistic regression model, and calculate validation accuracy rounded to 2 decimals.

- 0.55
- 0.65
- 0.75
- 0.85

In [24]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(
    solver='liblinear', C=1.0, max_iter=1000, random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
original_accuracy = (y_val == converted_decision).mean()
round(float(original_accuracy), 2)

0.65

## Question 5

Use feature elimination to find the feature with the smallest accuracy difference.

- `lead_source`
- `number_of_courses_viewed`
- `interaction_count`

In [25]:
scores = {}
features = categorical + numerical

for feature in features:
    subset = [name for name in features if name != feature]

    dv_feature = DictVectorizer(sparse=False)

    train_dict = df_train[subset].to_dict(orient='records')
    X_train_feature = dv_feature.fit_transform(train_dict)

    val_dict = df_val[subset].to_dict(orient='records')
    X_val_feature = dv_feature.transform(val_dict)

    model = LogisticRegression(
        solver='liblinear', C=1.0, max_iter=1000, random_state=42
    )
    model.fit(X_train_feature, y_train)

    y_pred = model.predict_proba(X_val_feature)[:, 1]
    converted_decision = (y_pred >= 0.5)
    accuracy = (y_val == converted_decision).mean()
    scores[feature] = original_accuracy - accuracy

scores

{'lead_source': np.float64(0.0030000000000000027),
 'industry': np.float64(0.0),
 'employment_status': np.float64(0.0),
 'location': np.float64(0.0),
 'annual_income': np.float64(-0.07899999999999996),
 'number_of_courses_viewed': np.float64(0.0020000000000000018),
 'interaction_count': np.float64(0.04400000000000004),
 'lead_score': np.float64(0.0010000000000000009)}

## Question 6

Train regularized logistic regression models for each `C`, calculate validation accuracy, and round to 3 decimals. If tied, choose the smallest `C`.

- 0.000001
- 0.00001
- 0.0001
- 0.001

In [26]:
scores = {}

for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(
        solver='liblinear', C=c, max_iter=1000, random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)
    scores[c] = round((y_val == converted_decision).mean(), 3)

scores

{1e-06: np.float64(0.598),
 1e-05: np.float64(0.598),
 0.0001: np.float64(0.613),
 0.001: np.float64(0.645)}